# Bloco 2 — Preparação dos Dados

Carregamento e transformação dos CSVs do dataset semi-sintético nas estruturas que suportam o modelo de otimização.

- Documenta a estrutura do dataset (10 tabelas)
- Restringe ao contexto de Housekeeping (turnos T1 e T2)
- Constrói os conjuntos e parâmetros base do modelo: `C`, `D`, `T`, `F`, `N_min`, `Disp`, `Qual`, ...

> Esta lógica é reutilizada pelo módulo optimizer.py

## 1. Configuração

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data" / "synthetic"

# Carregar os 10 CSVs
colaboradores       = pd.read_csv(DATA_DIR / "colaboradores.csv")
turnos              = pd.read_csv(DATA_DIR / "turnos.csv")
cobertura           = pd.read_csv(DATA_DIR / "cobertura_necessidades.csv")
disponibilidade     = pd.read_csv(DATA_DIR / "disponibilidade.csv")
qualificacoes       = pd.read_csv(DATA_DIR / "qualificacoes.csv")
preferencias        = pd.read_csv(DATA_DIR / "preferencias.csv")
atividade           = pd.read_csv(DATA_DIR / "atividade_hotel.csv")
tipologias_quarto   = pd.read_csv(DATA_DIR / "tipologias_quarto.csv")
atividade_tipologia = pd.read_csv(DATA_DIR / "atividade_tipologia.csv")
recursos_externos   = pd.read_csv(DATA_DIR / "recursos_externos.csv")

# Resumo do dataset
tabelas = {
    "colaboradores": colaboradores, "turnos": turnos,
    "cobertura_necessidades": cobertura, "disponibilidade": disponibilidade,
    "qualificacoes": qualificacoes, "preferencias": preferencias,
    "atividade_hotel": atividade, "tipologias_quarto": tipologias_quarto,
    "atividade_tipologia": atividade_tipologia, "recursos_externos": recursos_externos,
}
pd.DataFrame(
    [{"Tabela": nome, "Linhas": df.shape[0], "Colunas": df.shape[1]} for nome, df in tabelas.items()]
)

,Tabela,Linhas,Colunas
0,colaboradores,20,19
1,turnos,3,13
2,cobertura_necessidades,3285,8
3,disponibilidade,20075,12
4,qualificacoes,75,8
5,preferencias,60,5
6,atividade_hotel,365,14
7,tipologias_quarto,5,10
8,atividade_tipologia,1825,9
9,recursos_externos,5,13


## 2. Restrição ao contexto de Housekeeping

O modelo cobre apenas os turnos T1 e T2. A tabela abaixo mostra o efeito da filtragem nas quatro tabelas afectadas.

In [2]:
antes = {
    "turnos": len(turnos), "cobertura": len(cobertura),
    "disponibilidade": len(disponibilidade), "preferencias": len(preferencias),
}

turnos          = turnos[turnos["turno_id"].isin(["T1", "T2"])].copy()
cobertura       = cobertura[cobertura["turno_id"].isin(["T1", "T2"])].copy()
disponibilidade = disponibilidade[disponibilidade["turno_id"].isin(["T1", "T2"])].copy()
preferencias    = preferencias[preferencias["turno_id"].isin(["T1", "T2"])].copy()

pd.DataFrame([
    {"Tabela": t, "Antes": antes[t], "Depois": len(df), "Removidas": antes[t] - len(df)}
    for t, df in [("turnos", turnos), ("cobertura", cobertura),
                  ("disponibilidade", disponibilidade), ("preferencias", preferencias)]
])

,Tabela,Antes,Depois,Removidas
0,turnos,3,2,1
1,cobertura,3285,2190,1095
2,disponibilidade,20075,14600,5475
3,preferencias,60,40,20


## 3. Conjuntos base

Os quatro conjuntos principais do modelo: colaboradores `C`, dias `D`, turnos `T` e funções `F`.

In [3]:
cobertura["data"]           = pd.to_datetime(cobertura["data"])
disponibilidade["data"]     = pd.to_datetime(disponibilidade["data"])
atividade["data"]           = pd.to_datetime(atividade["data"])
atividade_tipologia["data"] = pd.to_datetime(atividade_tipologia["data"])

C = sorted(colaboradores.loc[colaboradores["ativo"] == 1, "colaborador_id"].unique().tolist())
D = sorted(cobertura["data"].dt.strftime("%Y-%m-%d").unique().tolist())
T = sorted(turnos.loc[turnos["ativo"] == 1, "turno_id"].unique().tolist())
F = sorted(cobertura["funcao"].unique().tolist())

print(f"C : {len(C)} colaboradores activos")
print(f"D : {len(D)} dias  ({D[0]} → {D[-1]})")
print(f"T : {T}")
print(f"F : {F}")

C : 20 colaboradores activos
D : 365 dias  (2025-01-01 → 2025-12-31)
T : ['T1', 'T2']
F : ['Auxiliar de limpeza', 'Empregada de andares', 'Supervisora']


## 4. Agrupamento semanal

`S` e `D_s` suportam as restrições semanais do modelo (ex: máximo de dias trabalhados por semana).

In [4]:
datas_df = pd.DataFrame({"data": pd.to_datetime(D)})
datas_df["semana_id"] = (
    datas_df["data"].dt.isocalendar().year.astype(str)
    + "-W"
    + datas_df["data"].dt.isocalendar().week.astype(str).str.zfill(2)
)
S   = sorted(datas_df["semana_id"].unique().tolist())
D_s = {
    s: datas_df.loc[datas_df["semana_id"] == s, "data"].dt.strftime("%Y-%m-%d").tolist()
    for s in S
}
print(f"S : {len(S)} semanas  (ex: {S[:3]})")

S : 53 semanas  (ex: ['2025-W01', '2025-W02', '2025-W03'])


## 5. Parâmetros do modelo

Transformação das tabelas em dicionários indexados pelas chaves do modelo.

**Parâmetros de cobertura** — necessidades mínimas e ideais por slot `(d, t, f)`  
**Parâmetros de turnos** — duração e flag de turno nocturno  
**Parâmetros de colaboradores** — custo, horas, dias e disponibilidade  
**Preferências** — score de preferência de turno e folga preferida

In [5]:
# ── Cobertura ────────────────────────────────────────────────────────────────
N_min = {
    (row["data"].strftime("%Y-%m-%d"), row["turno_id"], row["funcao"]): row["colaboradores_minimos"]
    for _, row in cobertura.iterrows()
}
N_ideal = {
    (row["data"].strftime("%Y-%m-%d"), row["turno_id"], row["funcao"]): row["colaboradores_ideais"]
    for _, row in cobertura.iterrows()
}

# ── Turnos ────────────────────────────────────────────────────────────────────
H_t     = {row["turno_id"]: row["duracao_horas"] for _, row in turnos.iterrows()}
Night_t = {row["turno_id"]: row["turno_noite"]   for _, row in turnos.iterrows()}

# ── Colaboradores ─────────────────────────────────────────────────────────────
Cost_c     = {row["colaborador_id"]: row["custo_hora"]               for _, row in colaboradores.iterrows()}
H_dia_c    = {row["colaborador_id"]: row["horas_max_dia"]            for _, row in colaboradores.iterrows()}
Dias_sem_c = {row["colaborador_id"]: row["dias_max_trabalho_semana"] for _, row in colaboradores.iterrows()}
K_c        = {row["colaborador_id"]: row["max_dias_consecutivos"]    for _, row in colaboradores.iterrows()}

# ── Disponibilidade e preferências ────────────────────────────────────────────
Disp = {
    (row["colaborador_id"], row["data"].strftime("%Y-%m-%d"), row["turno_id"]): row["disponivel"]
    for _, row in disponibilidade.iterrows()
}
Pref = {
    (row["colaborador_id"], row["turno_id"]): row["score_preferencia_turno"]
    for _, row in preferencias.iterrows()
}
FolgaPref_c = {
    row["colaborador_id"]: row["folga_preferida_dow"]
    for _, row in preferencias.drop_duplicates(subset=["colaborador_id"]).iterrows()
}

## 6. Parâmetro de qualificação `Qual(c, f)`

Mapeia cada função operacional para a qualificação correspondente e constrói `Qual(c,f) ∈ {0,1}`.

In [6]:
map_funcao_qual = {
    "Empregada de andares": "quartos_saida",
    "Supervisora":          "supervisao",
    "Governanta":           "supervisao",
    "Auxiliar de limpeza":  "areas_publicas",
}

qual_set = {
    (row["colaborador_id"], row["qualificacao"])
    for _, row in qualificacoes.iterrows()
    if row["pode_executar"] == 1
}

Qual = {
    (c, f): 1 if (c, map_funcao_qual.get(f)) in qual_set else 0
    for c in C for f in F
}

print(f"Pares (c,f) totais      : {len(Qual)}")
print(f"Pares (c,f) qualificados: {sum(Qual.values())}")

Pares (c,f) totais      : 60
Pares (c,f) qualificados: 37


## 7. Validação dos parâmetros

Resumo dos principais parâmetros construídos — dimensão e exemplo de entrada.

In [7]:
pd.DataFrame([
    {"Parâmetro": "N_min",       "Entradas": len(N_min),   "Exemplo de chave": list(N_min.keys())[0],   "Exemplo de valor": list(N_min.values())[0]},
    {"Parâmetro": "N_ideal",     "Entradas": len(N_ideal), "Exemplo de chave": list(N_ideal.keys())[0], "Exemplo de valor": list(N_ideal.values())[0]},
    {"Parâmetro": "H_t",         "Entradas": len(H_t),     "Exemplo de chave": list(H_t.keys())[0],     "Exemplo de valor": list(H_t.values())[0]},
    {"Parâmetro": "Cost_c",      "Entradas": len(Cost_c),  "Exemplo de chave": list(Cost_c.keys())[0],  "Exemplo de valor": list(Cost_c.values())[0]},
    {"Parâmetro": "Disp",        "Entradas": len(Disp),    "Exemplo de chave": list(Disp.keys())[0],    "Exemplo de valor": list(Disp.values())[0]},
    {"Parâmetro": "Pref",        "Entradas": len(Pref),    "Exemplo de chave": list(Pref.keys())[0],    "Exemplo de valor": list(Pref.values())[0]},
    {"Parâmetro": "Qual",        "Entradas": len(Qual),    "Exemplo de chave": list(Qual.keys())[0],    "Exemplo de valor": list(Qual.values())[0]},
])

,Parâmetro,Entradas,Exemplo de chave,Exemplo de valor
0,N_min,2190,"(2025-01-01, T1, Empregada de andares)",4.00
1,N_ideal,2190,"(2025-01-01, T1, Empregada de andares)",5.00
2,H_t,2,T1,7.00
3,Cost_c,20,C001,13.05
4,Disp,14600,"(C001, 2025-01-01, T1)",1.00
5,Pref,40,"(C001, T1)",0.00
6,Qual,60,"(C001, Auxiliar de limpeza)",0.00
